In [1]:
import pandas as pd
import numpy as np
import os
import math
import warnings
import pickle
import hashlib
from collections import Counter
from itertools import islice

import scipy
from scipy import stats
from scipy.stats import linregress, pearsonr, percentileofscore

import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt
import shap

from tqdm.notebook import tqdm
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

from sklearn import metrics
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import check_random_state, resample

from skopt import BayesSearchCV
from skopt.space import Real, Integer

warnings.filterwarnings("ignore")

In [2]:
import numpy as np
from scipy.stats import percentileofscore
from sklearn.utils import check_random_state
from itertools import islice

def permtest(x, y, statistic="mean", max_samples=5000, random_state=42):
  """
  Conducts a permutation test between two independent samples.
  Parameters
  ----------
  x : ndarray
    First set of datapoints.
  y : ndarray
    Second set of datapoints.
  statistic : str or callable, optional
    Function that takes in samples x and y and reports
    statistic of interest. By default, "mean" reports
    the difference of sample means. List of default
    options are ("mean", "median").
  max_samples : int, optional.
    Maximum number of label permutations to try.
  random_state : np.random.RandomState, int, or None.
    If specified, used to seed the random number generator to shuffle the ordering of the datapoints.
  """

  # Initialize random state and function of interest
  rs = check_random_state(random_state)
  stat_func = _get_stat_func(statistic)

  # Concatenate samples
  xy = np.concatenate((x, y))
  n_x = x.size

  # Observed statistic
  observed_stat = stat_func(x, y)

  # Set the number of permutations to compute
  n_perms = max(max_samples, 100000) # Limit the number of permutations to a manageable number (e.g., 10000)

  # Allocate space for computed statistics
  shuffled_stats = np.empty(n_perms)

  # Print coverage
  print(f"Computing permutations...")

  # Sampling random permutations
  for i in range(n_perms):
    rs.shuffle(xy)
    x_ = xy[:n_x]
    y_ = xy[n_x:]
    shuffled_stats[i] = stat_func(x_, y_)

  # Compute a two-sided p-value. We take the smallest
  # percentile and then multiply by two.
  pval = 2 * min(
    percentileofscore(shuffled_stats, observed_stat, kind="rank") / 100,
    1 - percentileofscore(shuffled_stats, observed_stat, kind="rank") / 100
  )

  return pval

def _get_stat_func(name_or_func):
  """
  Instantiates functions that compute default statistics of interest.
  """
  # If specified function is callable
  if not isinstance(name_or_func, str):
    if callable(name_or_func):
      return name_or_func
    else:
      raise ValueError(
        "`statistic` should be a string like ('mean', 'median')"
        " or a function that takes in samples x, y and returns"
        " the statistic of interest."
      )

  # Default functions
  if name_or_func == "mean":
    return lambda x, y: np.mean(x) - np.mean(y)
  elif name_or_func == "median":
    return lambda x, y: np.median(x) - np.median(y)
  else:
    raise ValueError("Did not recognize statistic.")

In [3]:
import numpy as np

# Define the pooled function to calculate the combined value of a specific measure
def pooled(x, y, func):
  nx = len(x)
  ny = len(y)
  return np.sqrt(((nx - 1) * func(x) ** 2 + (ny - 1) * func(y) ** 2) / (nx + ny - 2))

# Define the median, using the median of quantiles
def hdmedian(x):
  return np.median(x)

# Define the hdmad function to calculate the adjusted MAD (Median Absolute Deviation)
def hdmad(x):
  return 1.4826 * hdmedian(np.abs(x - hdmedian(x)))

# Calculate the pooled MAD between two datasets x and y
def phdmad(x, y):
  return pooled(x, y, hdmad)

# Define the gammaEffectSize function to calculate the gamma effect size
def gamma_effect_size(x, y, prob):
  hdquantile_x = np.percentile(x, prob * 100)
  hdquantile_y = np.percentile(y, prob * 100)
  return (hdquantile_y - hdquantile_x) / phdmad(x, y)

# Usage example
x = np.array([1, 2, 3, 4, 5])
y = np.array([3, 4, 5, 6, 7])

# Calculate the gamma effect size
prob = 0.5 # median
gamma_effect = gamma_effect_size(x, y, prob)
print(f'Gamma effect size: {gamma_effect}')

In [ ]:
def mean_directional_accuracy(y_true, y_pred):
    
    differences = np.array(y_pred) - np.array(y_true) 
    signs = np.sign(differences)
    mde = np.mean(signs)
    return mde

def mean_absolute_error(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    absolute_errors = np.abs(y_pred - y_true)
    mae = np.mean(absolute_errors)
    
    return mae

def coef_pval(coef_array_mean_, X_, y_, y_p):

    n = X_.shape[0]
    t = coef_tval(coef_array_mean_, X_, y_, y_p)
    p = 2 * (1 - scipy.stats.t.cdf(abs(t), n - 1))
    return p


def coef_tval(coef_array_mean_, X_, y_, y_p):
    
    '''
        coef_tval for OLS of statsmodels
    '''
    
    a = np.array(coef_array_mean_[0][0]/ coef_se(X_, y_, y_p)[0])
    b = np.array(coef_array_mean_[1::].flatten() / coef_se(X_, y_, y_p)[1:])
    return np.append(a, b)


def coef_se(X_, y_, y_p):
    
    '''
        coef_se for OLS of statsmodels
    '''
    n = X_.shape[0]
    
    X1 = np.hstack((np.ones((n, 1)), np.matrix(X_)))
    se_matrix = scipy.linalg.sqrtm(
        metrics.mean_squared_error(y_, y_p) *
        np.linalg.inv(X1.T * X1)
    )
    return np.diagonal(se_matrix)

def directional_accuracy(predicted_values, true_values):

    predicted_values = np.array(predicted_values)
    true_values = np.array(true_values)    

    difference_direction = np.sign(predicted_values - true_values)    
    correct_direction_count = np.sum(difference_direction == 1) + np.sum(difference_direction == -1)
    
    directional_accuracy_score = correct_direction_count / len(predicted_values)
    
    return directional_accuracy_score

In [8]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import linregress
import numpy as np
import pandas as pd
import math
import warnings
from skopt import BayesSearchCV
from skopt.space import Real, Integer

def Regression_GBR_nested(X, y, outer_splits=10, inner_splits=5, shaps_comp=False):
  warnings.filterwarnings("ignore")

  scaler = MinMaxScaler((0.05, 0.95))

  outer_cv = KFold(n_splits=outer_splits, shuffle=True, random_state=42)
  param_grid = {
    'n_estimators': Integer(50, 500),          # Number of trees
    'learning_rate': Real(0.01, 0.5, prior='log-uniform'),  # Learning rate
    'max_depth': Integer(1, 5),               # Maximum depth of the trees
    'min_samples_split': Integer(2, 10),       # Minimum samples to split a node
    'min_samples_leaf': Integer(1, 10),        # Minimum samples in a leaf
  }

  y_labels = []
  y_predicts = []
  r_squared_l = []
  mse_l = []
  rmse_l = []

  results_labels_df = pd.DataFrame(columns=['y_labels', 'y_pred', 'GAP', 'GAP_corrected', 'ID'])

  coef_array = np.zeros([X.shape[1] + 1, outer_splits])
  lista_vars = list(X.columns)

  for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X)):

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print(f"Fold: {fold}")
    
    scaling_data = scaler.fit_transform(X_train)
    X_train = pd.DataFrame(scaling_data, columns= X_train.columns, index = X_train.index)

    scaling_data = scaler.transform(X_test)
    X_test = pd.DataFrame(scaling_data, columns= X_test.columns, index = X_test.index)
    
    model = GradientBoostingRegressor(random_state=42)
    inner_cv = KFold(n_splits=inner_splits, shuffle=True, random_state=42)
    
    bayes_search = BayesSearchCV(
      estimator=model,
      search_spaces=param_grid,
      n_iter=3,              
      scoring='r2',  
      cv=inner_cv,                   
      n_jobs=-1,             
      random_state=42
    )
    bayes_search.fit(X_train, y_train)

    best_model = bayes_search.best_estimator_

    y_pred = best_model.predict(X_test)
    gap_test = y_pred - y_test
    gap_train = best_model.predict(X_train) - y_train

    slope, intercept, _, _, _ = linregress(y_train, gap_train)
    corrected_gap = gap_test - (slope * y_test + intercept)

    result = np.column_stack((y_test, y_pred, gap_test, corrected_gap))
    temp_df = pd.DataFrame(result, columns=['y_labels', 'y_pred', 'GAP', 'GAP_corrected'])
    temp_df['ID'] = X_test.index
    results_labels_df = pd.concat([results_labels_df, temp_df], ignore_index=True)

    y_labels.extend(y_test)
    y_predicts.extend(y_pred)

    r_squared_l.append(r2_score(y_test, y_pred))
    mse_l.append(mean_squared_error(y_test, y_pred))
    rmse_l.append(math.sqrt(mse_l[-1]))

    coef_array[0, fold] = np.nan
    coef_array[1:, fold] = best_model.feature_importances_

  # Final metrics
  y_labels = np.array(y_labels)
  y_predicts = np.array(y_predicts)

  r_squared = r2_score(y_labels, y_predicts)
  r_squared_ = np.mean(r_squared_l)
  n = len(y_labels)
  p = X.shape[1]
  k = p - 1

  r_squared_adj = 1 - (1 - r_squared) * (n - 1) / (n - k - 1)
  mse = np.mean(mse_l)
  rmse = np.mean(rmse_l)
  mae = mean_absolute_error(y_labels, y_predicts)
  F = (r_squared / p) / ((1 - r_squared) / (n - p - 1))
  p_value = np.round(scipy.stats.f.sf(F, n, (n - p - 1)), 15)
  F2 = r_squared / (1 - r_squared)

  # Mean coefficients
  coef_df = pd.DataFrame(index=['_intercept'] + lista_vars,
              columns=['Estimate mean', 'Estimate std', 't value', 'p value'])

  coef_df['Estimate mean'] = coef_array.mean(axis=1)
  coef_df['Estimate std'] = coef_array.std(axis=1)

  coef_df.loc['_intercept', 'R2'] = r_squared
  coef_df.loc['_intercept', 'R2 adj'] = r_squared_adj
  coef_df.loc['_intercept', 'R2 [+-]'] = np.std(r_squared_l)
  coef_df.loc['_intercept', 'F2'] = F2
  coef_df.loc['_intercept', 'mse'] = mse
  coef_df.loc['_intercept', 'mse [+-]'] = np.std(mse_l)
  coef_df.loc['_intercept', 'rmse'] = rmse
  coef_df.loc['_intercept', 'rmse [+-]'] = np.std(rmse_l)
  coef_df.loc['_intercept', 'outcome var'] = np.var(y)
  coef_df.loc['_intercept', 'F'] = F
  coef_df.loc['_intercept', 'F-p_value'] = p_value
  try:
    mde = mean_directional_accuracy(y_labels, y_predicts)
  except NameError:
    mde = np.nan
  coef_df.loc['_intercept', 'MDE'] = mde
  coef_df.loc['_intercept', 'MAE'] = mae

  results_labels_df['y_pred_corrected'] = results_labels_df['y_labels'] + results_labels_df['GAP_corrected']
  results_labels_df = results_labels_df[['ID','y_labels', 'y_pred', 'GAP', 'GAP_corrected', 'y_pred_corrected']]

  if shaps_comp:
    import shap
    explainer = shap.Explainer(best_model, X)
    return [coef_df, r_squared_, results_labels_df, explainer, outer_cv.split(X)]
  else:
    return [coef_df, r_squared_, results_labels_df, outer_cv.split(X)]

In [9]:
from sklearn.model_selection import KFold
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import linregress
from scipy import stats
import numpy as np
import pandas as pd
import math
import warnings
from skopt import BayesSearchCV
from skopt.space import Real, Integer

def Regression_GBR_nested(X, y, dsmcase, outer_splits=10, inner_splits=5, shaps_comp=False, control_value=1):
  """
  Nested-CV for GBR. GAP-correction is fit ONLY on training controls (dsmcase==control_value)
  and then applied to the test fold.
  X: DataFrame (features)
  y: Series (target)
  dsmcase: Series aligned with X/y; 1=controls by default (set control_value otherwise)
  """
  warnings.filterwarnings("ignore")

  # --- Scaler ---
  scaler = MinMaxScaler((0.05, 0.95))

  # --- CV and search space ---
  outer_cv = KFold(n_splits=outer_splits, shuffle=True, random_state=42)
  param_grid = {
    'n_estimators': Integer(50, 500),
    'learning_rate': Real(0.01, 0.5, prior='log-uniform'),
    'max_depth': Integer(1, 5),
    'min_samples_split': Integer(2, 10),
    'min_samples_leaf': Integer(1, 10),
  }

  # --- Collectors ---
  y_labels = []
  y_predicts = []
  r_squared_l, mse_l, rmse_l = [], [], []
  results_labels_df = pd.DataFrame(columns=['y_labels', 'y_pred', 'GAP', 'GAP_corrected', 'ID'])

  coef_array = np.zeros([X.shape[1] + 1, outer_splits])
  lista_vars = list(X.columns)

  # --- Outer CV ---
  for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X)):
    print(f"Fold: {fold}")

    # Split
    X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
    dsm_train, dsm_test = dsmcase.iloc[train_idx].copy(), dsmcase.iloc[test_idx].copy()

    # Scale
    X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns,  index=X_test.index)

    # Model + inner CV (BayesSearch)
    model = GradientBoostingRegressor(random_state=42)
    inner_cv = KFold(n_splits=inner_splits, shuffle=True, random_state=42)

    bayes_search = BayesSearchCV(
      estimator=model,
      search_spaces=param_grid,
      n_iter=3,
      scoring='r2',
      cv=inner_cv,
      n_jobs=-1,
      random_state=42
    )
    bayes_search.fit(X_train, y_train)
    best_model = bayes_search.best_estimator_

    # Predictions
    y_pred_test = best_model.predict(X_test)
    y_pred_train = best_model.predict(X_train)

    gap_test = y_pred_test - y_test
    gap_train = y_pred_train - y_train

    # ----- GAP correction trained ONLY on training CONTROLS -----
    # mask of controls in train
    mask_ctrl_train = (dsm_train == control_value)
    y_train_ctrl  = y_train[mask_ctrl_train]
    gap_train_ctrl = gap_train[mask_ctrl_train]

    # Guard against too-few controls or degenerate variance
    slope, intercept = 0.0, 0.0
    y_ctrl_vals = np.asarray(y_train_ctrl, dtype=float)
    gap_ctrl_vals = np.asarray(gap_train_ctrl, dtype=float)

    # Need at least 2 points and non-zero variance in y to fit a line
    if np.sum(~np.isnan(y_ctrl_vals) & ~np.isnan(gap_ctrl_vals)) >= 2 and np.nanstd(y_ctrl_vals) > 0:
      slope, intercept, _, _, _ = linregress(y_ctrl_vals, gap_ctrl_vals)
    else:
      # Fallback: remove constant bias using mean gap on controls
      if np.isfinite(np.nanmean(gap_ctrl_vals)):
        intercept = float(np.nanmean(gap_ctrl_vals))
      else:
        intercept = 0.0
      slope = 0.0

    # Apply correction to TEST fold
    corrected_gap = gap_test - (slope * y_test + intercept)

    # Save per-sample results
    tmp = pd.DataFrame({
      'y_labels': y_test.values,
      'y_pred':  y_pred_test,
      'GAP':   gap_test,
      'GAP_corrected': corrected_gap
    }, index=y_test.index)
    tmp['ID'] = X_test.index
    results_labels_df = pd.concat([results_labels_df, tmp.reset_index(drop=True)], ignore_index=True)

    # Metrics per fold
    y_labels.extend(y_test.values)
    y_predicts.extend(y_pred_test)
    r_squared_l.append(r2_score(y_test, y_pred_test))
    mse_l.append(mean_squared_error(y_test, y_pred_test))
    rmse_l.append(math.sqrt(mse_l[-1]))

    # Feature importances (GBR)
    coef_array[0, fold] = np.nan
    coef_array[1:, fold] = best_model.feature_importances_

  # --- Aggregate metrics (across all test folds) ---
  y_labels = np.array(y_labels)
  y_predicts = np.array(y_predicts)

  r_squared = r2_score(y_labels, y_predicts)
  r_squared_mean = float(np.mean(r_squared_l))
  n = len(y_labels)
  p = X.shape[1] # number of predictors (no intercept in X)

  # Adjusted R^2 with df = n - p - 1
  r_squared_adj = 1 - (1 - r_squared) * (n - 1) / max(1, (n - p - 1))

  mse = float(np.mean(mse_l))
  rmse = float(np.mean(rmse_l))
  mae = float(mean_absolute_error(y_labels, y_predicts))

  # Simple F-stat approximation (same df as OLS)
  if (n - p - 1) > 0 and (1 - r_squared) > 0 and p > 0:
    F = (r_squared / p) / ((1 - r_squared) / (n - p - 1))
    p_value = float(np.round(stats.f.sf(F, p, (n - p - 1)), 15))
    F2 = r_squared / (1 - r_squared)
  else:
    F, p_value, F2 = np.nan, np.nan, np.nan

  # --- Coefficients/Importances summary ---
  coef_df = pd.DataFrame(index=['_intercept'] + lista_vars,
              columns=['Estimate mean', 'Estimate std', 't value', 'p value'])

  coef_df['Estimate mean'] = coef_array.mean(axis=1)
  coef_df['Estimate std'] = coef_array.std(axis=1)

  # Global metrics in the intercept row
  coef_df.loc['_intercept', 'R2']    = r_squared
  coef_df.loc['_intercept', 'R2 adj']  = r_squared_adj
  coef_df.loc['_intercept', 'R2 [+-]']  = np.std(r_squared_l)
  coef_df.loc['_intercept', 'F2']    = F2
  coef_df.loc['_intercept', 'mse']    = mse
  coef_df.loc['_intercept', 'mse [+-]'] = np.std(mse_l)
  coef_df.loc['_intercept', 'rmse']   = rmse
  coef_df.loc['_intercept', 'rmse [+-]'] = np.std(rmse_l)
  coef_df.loc['_intercept', 'outcome var'] = np.var(y)

  coef_df.loc['_intercept', 'F']     = F
  coef_df.loc['_intercept', 'F-p_value'] = p_value

  # Optional: mean directional accuracy if your function exists
  try:
    mde = mean_directional_accuracy(y_labels, y_predicts)
  except NameError:
    mde = np.nan
  coef_df.loc['_intercept', 'MDE'] = mde
  coef_df.loc['_intercept', 'MAE'] = mae

  # Add corrected predictions (y_pred_corrected = y + GAP_corrected)
  results_labels_df['y_pred_corrected'] = results_labels_df['y_labels'] + results_labels_df['GAP_corrected']
  results_labels_df = results_labels_df[['ID','y_labels','y_pred','GAP','GAP_corrected','y_pred_corrected']]

  if shaps_comp:
    import shap
    explainer = shap.Explainer(best_model, X)
    return [coef_df, r_squared_mean, results_labels_df, explainer, outer_cv.split(X)]
  else:
    return [coef_df, r_squared_mean, results_labels_df, outer_cv.split(X)]


## Functions

In [10]:
import os
import pickle
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

def get_best_features_sfs(data, vars_list, target_col="Age",
             pkl_path="best_features_w-Barthel-Cognition_v01.pkl",
             force=True):
  if (not force) and os.path.exists(pkl_path):
    with open(pkl_path, "rb") as f:
      best_features = pickle.load(f)
    print("Loaded best_features from pickle:", best_features)
  else:

    gbr = HistGradientBoostingRegressor()

    # Apply forward SFS using R2 as metric
    sfs = SFS(gbr,
         k_features='best', # or a fixed number like 5 or 10 if preferred
         forward=True,
         floating=False,
         scoring='r2',
         cv=5,
         n_jobs=-1,
         verbose=2)

    # Execute SFS
    sfs = sfs.fit(data[vars_list], data[target_col])

    # Get the best selected features
    best_features = list(sfs.k_feature_names_)
    with open(pkl_path, "wb") as f:
      pickle.dump(best_features, f)
    print("Computed and saved best_features:", best_features)

  return best_features

In [11]:
def run_nested_cv_hgbr(data_, best_features,
            y_col="Age", diag_col="anydem"):
  from sklearn.model_selection import KFold, GridSearchCV
  from sklearn.pipeline import Pipeline
  from sklearn.preprocessing import MinMaxScaler
  from sklearn.ensemble import HistGradientBoostingRegressor
  from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
  from scipy.stats import linregress
  import numpy as np
  import pandas as pd
  import shap
  from sklearn.inspection import permutation_importance

  # Variables
  y = data_[y_col]
  X_selected = data_[best_features] # selected variables

  # Outer CV
  kf = KFold(n_splits=10, shuffle=True, random_state=42)

  # Hyperparameters for Gradient Boosting
  param_grid = {
    "model__max_iter": [300, 400, 500, 600],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.05, 0.1]
  }

  # Results
  r2_scores = []
  r2_adj_scores = [] # nuevo
  y_true_all = []
  y_pred_all = []
  results_labels_df = pd.DataFrame(columns=['y_labels', 'y_pred', 'GAP', 'GAP_corrected', 'ID'])

  # SHAP accumulator
  shap_values_sum = pd.Series(0, index=X_selected.columns)
  perm_values_sum = pd.Series(0, index=X_selected.columns)

  p = X_selected.shape[1] # number of predictors (constant)

  # Nested CV
  for fold, (train_idx, test_idx) in enumerate(kf.split(X_selected)):
    print(f"Fold {fold + 1} (Nested CV)")

    X_train, X_test = X_selected.iloc[train_idx], X_selected.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Pipeline
    pipeline = Pipeline([
      ("scaler", MinMaxScaler((0.05, 0.95))),
      ("model", HistGradientBoostingRegressor(random_state=42))
    ])

    # Internal GridSearch
    grid_search = GridSearchCV(
      estimator=pipeline,
      param_grid=param_grid,
      scoring="r2",
      cv=5,
      n_jobs=-1,
      verbose=0
    )

    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_

    # Evaluate
    y_pred = best_model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)

    # Adjusted R2 per fold
    n_test = len(y_test)
    if (n_test - p - 1) > 0:
      r2_adj = 1 - (1 - r2) * (n_test - 1) / (n_test - p - 1)
    else:
      r2_adj = np.nan
    r2_adj_scores.append(r2_adj)

    print(f"R² fold {fold + 1}: {r2:.4f} | R² adj: {r2_adj:.4f} (best hyperparameters: {grid_search.best_params_})")

    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)

    # GAP
    gap_test = y_pred - y_test
    gap_train_all = best_model.predict(X_train) - y_train

    # Filter only CN subjects in train for regression
    train_ids = X_train.index
    diag_train = data_.loc[train_ids, diag_col]
    cn_mask = diag_train == 0

    slope, intercept, _, _, _ = linregress(y_train[cn_mask], gap_train_all[cn_mask])
    corrected_gap = gap_test - (slope * y_test + intercept)

    # SHAP values for this fold
    explainer = shap.Explainer(best_model.named_steps['model'], X_train)
    shap_values = explainer(X_test, check_additivity=False)
    shap_values_mean = np.abs(shap_values.values).mean(axis=0)
    shap_values_sum += pd.Series(shap_values_mean, index=X_selected.columns)

    perm_importance = permutation_importance(
      best_model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1
    )

    # Average decrease in R²
    perm_values_mean = perm_importance.importances_mean
    perm_values_sum += pd.Series(perm_values_mean, index=X_selected.columns)

    # Save results of this fold
    result = np.column_stack((y_test, y_pred, gap_test, corrected_gap))
    temp_df = pd.DataFrame(result, columns=['y_labels', 'y_pred', 'GAP', 'GAP_corrected'])
    temp_df['ID'] = X_test.index
    results_labels_df = pd.concat([results_labels_df, temp_df], ignore_index=True)

  # r (Pearson correlation)
  r = np.corrcoef(y_true_all, y_pred_all)[0, 1]

  # RMSE
  rmse = np.sqrt(mean_squared_error(y_true_all, y_pred_all))

  # MAE
  mae = mean_absolute_error(y_true_all, y_pred_all)

  # Cohen's f²: f² = R² / (1 - R²)
  f2 = r2 / (1 - r2) if r2 < 1 else np.inf

  # Global R2 and global adjusted R2
  r2_global = r2_score(y_true_all, y_pred_all)
  n_global = len(y_true_all)
  if (n_global - p - 1) > 0:
    r2_adj_global = 1 - (1 - r2_global) * (n_global - 1) / (n_global - p - 1)
  else:
    r2_adj_global = np.nan

  # Final results
  print("\nR² per fold (Nested CV):", r2_scores)
  print("Adjusted R² per fold (Nested CV):", r2_adj_scores)
  print("Average R²:", np.mean(r2_scores), np.std(r2_scores))
  print("Average adjusted R²:", np.nanmean(r2_adj_scores), np.nanstd(r2_adj_scores))
  print("Global R² (all predictions):", r2_global)
  print("Global adjusted R²:", r2_adj_global)

  print("Cohen's f²:", f2)
  print("r (correlation):", r)
  print("RMSE:", rmse)
  print("MAE:", mae)

  # Average SHAP importance
  shap_importance_avg = shap_values_sum / kf.get_n_splits()
  shap_importance_avg = shap_importance_avg.sort_values(ascending=False)
  print("\nAverage importance of SHAP values:")
  print(shap_importance_avg)

  # Average permutation importance
  perm_importance_avg = perm_values_sum / kf.get_n_splits()
  perm_importance_avg = perm_importance_avg.sort_values(ascending=False)
  print("\nAverage importance (Permutation Importance):")
  print(perm_importance_avg)

  return (r2_scores, r2_adj_scores, results_labels_df, shap_importance_avg, perm_importance_avg, r2_global, r2_adj_global)



In [ ]:
def nested_cv_flag_bad_subjects(data_, best_features, y_col="Age", diag_col="anydem"):
  from sklearn.model_selection import KFold, GridSearchCV
  from sklearn.pipeline import Pipeline
  from sklearn.preprocessing import MinMaxScaler
  from sklearn.ensemble import HistGradientBoostingRegressor
  from sklearn.metrics import r2_score
  from scipy.stats import linregress
  import numpy as np
  import pandas as pd

  # Variables
  y = data_[y_col]
  X_selected = data_[best_features]

  kf = KFold(n_splits=10, shuffle=True, random_state=42)

  param_grid = {
    "model__max_iter": [50, 100, 200],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.05, 0.1]
  }

  r2_scores = []
  y_true_all, y_pred_all = [], []
  results_labels_df = pd.DataFrame(columns=['y_labels', 'y_pred', 'GAP', 'GAP_corrected', 'ID'])
  suspected_bad_subjects = []

  # Nested CV
  for fold, (train_idx, test_idx) in enumerate(kf.split(X_selected)):
    print(f"Fold {fold + 1} (Nested CV)")

    X_train, X_test = X_selected.iloc[train_idx], X_selected.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    pipeline = Pipeline([
      ("scaler", MinMaxScaler((0.05, 0.95))),
      ("model", HistGradientBoostingRegressor(random_state=42))
    ])

    grid_search = GridSearchCV(
      estimator=pipeline,
      param_grid=param_grid,
      scoring="r2",
      cv=5,
      n_jobs=-1,
      verbose=0
    )

    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_

    y_pred = best_model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)

    print(f"R² fold {fold + 1}: {r2:.4f}")

    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)

    gap_test = y_pred - y_test
    gap_train = best_model.predict(X_train) - y_train
    slope, intercept, _, _, _ = linregress(y_train, gap_train)
    corrected_gap = gap_test - (slope * y_test + intercept)

    result = np.column_stack((y_test, y_pred, gap_test, corrected_gap))
    temp_df = pd.DataFrame(result, columns=['y_labels', 'y_pred', 'GAP', 'GAP_corrected'])
    temp_df['ID'] = X_test.index

    results_labels_df = pd.concat([results_labels_df, temp_df], ignore_index=True)

    # Identify subjects that greatly reduce R² in this fold
    if r2 < 0.25:
      individual_errors = np.abs(y_test - y_pred)
      threshold = np.percentile(individual_errors, 95) # top 10% worst
      bad_subjects_fold = X_test.index[individual_errors >= threshold].tolist()
      suspected_bad_subjects.extend(bad_subjects_fold)

  # Filter only those that are NOT CN
  non_cn_bad_subjects = [
    subj for subj in set(suspected_bad_subjects)
    if data_.loc[subj, diag_col] != 0
  ]

  print(f"Subjects identified (NOT CN) with negative impact: {len(non_cn_bad_subjects)}")

  # Create new dataframe without those NOT CN subjects
  data_filtered = data_.drop(index=non_cn_bad_subjects)
  print(f"New filtered dataframe: {data_filtered.shape}")

  # Final results
  print("\nAverage R²:", np.mean(r2_scores), np.std(r2_scores))
  print("Global R²:", r2_score(y_true_all, y_pred_all))

  return (data_filtered, non_cn_bad_subjects, r2_scores, results_labels_df)

## All vars model

### Load data

In [13]:
data = pd.read_parquet('../../Data/Wave2_ModelC.parquet')

In [14]:
from collections import Counter
Counter(data['prfrate'])

Counter({nan: 1,
         0.0: 7992,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         1.0: 375,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         nan: 1,
         

In [15]:
data = data[data['prfrate']<=2]
data.reset_index(inplace=True, drop = True)

In [16]:
data = data[data['age']<=101]
data.reset_index(inplace=True, drop = True)

In [18]:
data = data.rename(columns={'age':'Age'})

In [20]:
data['pulse_pressure'] = data['sbp_mean'] - data['dbp_mean']


In [21]:
vars_list = ['Sex_1F_2M', 'Education', 'Assets', 'Ataxia', 'Bradykinesia', 'Hypertension_1Y_0N', 'Heart_Disease_1Y_0N', 'Vision_problems', 'Audition_problems', 'pulse_pressure', 'orthostatic_drop', 'Stroke_1Y_0N', 'TIA_1Y_0N', 'Back_diseases', 'Medic_treated', 'Arthritis', 'Cough', 'Breathlessness', 'Breath_problems', 'Angina', 'Stomach_problems', 'Faints', 'Paralysis', 'Limiting_illnesses', 'Skin_disorder', 'Pain', 'Alcohol_1Y_0N', 'Physical_activities', 'Walk_1Y_0N', 'Exercise_increase', 'Medic_visits', 'Medications_1Y_0N']

In [23]:
missing_percentages = data[vars_list].isna().mean()

vars_list_clean = missing_percentages[missing_percentages <= 0.10].index.tolist()

print("Kept variables (<= 10% NaN):")
print(vars_list_clean)

print("\nRemoved variables (> 10% NaN):")
print(missing_percentages[missing_percentages > 0.10].index.tolist())

In [25]:
data.dropna(subset=vars_list + ['Age'], inplace=True)
data.reset_index(inplace=True, drop = True)

### SFS all subjects

In [31]:
best_features = get_best_features_sfs(data, vars_list,
                   target_col="Age",
                   pkl_path="SFS/best_features-wo-used-diag-vars-and-funct-vars_all-subjects-Incidence.pkl",
                   force=False)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of  23 | elapsed:   31.2s remaining:  2.5min
[Parallel(n_jobs=-1)]: Done  16 out of  23 | elapsed:   34.4s remaining:   15.0s
[Parallel(n_jobs=-1)]: Done  23 out of  23 | elapsed:   35.8s finished

[2026-01-11 19:10:10] Features: 1/23 -- score: 0.06638742761194931[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of  22 | elapsed:    3.3s remaining:   21.4s
[Parallel(n_jobs=-1)]: Done  15 out of  22 | elapsed:    8.6s remaining:    4.0s
[Parallel(n_jobs=-1)]: Done  22 out of  22 | elapsed:   10.9s finished

[2026-01-11 19:10:21] Features: 2/23 -- score: 0.09347639676900062[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  12 out of  21 | elapsed:   13.8s remaining:   10.3s
[Parallel(n_jobs=-1)]: Done  21 out of  21 | elapsed:   15.9s finished

[2026-01-11 19:1

Computed and saved best_features: ['Audition_problems', 'Physical_activities', 'Walk_1Y_0N']



[2026-01-11 19:14:12] Features: 23/23 -- score: 0.09299500901203728

In [32]:
## Ensure these are taken into account
best_features.append('Sex_1F_2M')
best_features.append('Education')

best_features = list(np.unique(best_features))

### BBAGs model-all subjects

In [ ]:
#!pip install scikit-optimize

#from scipy.stats import pointbiserialr
#
## Assuming 'Age' is continuous and 'Group' is binary 0/1
#corr, p_value = pointbiserialr(data['Diabetes_1Y_0N'], data['Age'])
#
#print(f"Point-biserial correlation: {corr:.3f}, p-value: {p_value:.3f}")

In [ ]:
r2_scores, r2_adj_scores, results_df, shap_imp, perm_imp, r2_global, r2_adj_global = run_nested_cv_hgbr(
  data_=data,
  best_features=best_features,
  y_col="Age",
  diag_col="anydem"
)

🔁 Fold 1 (Nested CV)
🏆 R² fold 1: 0.1759 | R² adj: 0.1710 (mejores hiperparámetros: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__max_iter': 400})
🔁 Fold 2 (Nested CV)
🏆 R² fold 2: 0.1841 | R² adj: 0.1792 (mejores hiperparámetros: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__max_iter': 400})
🔁 Fold 3 (Nested CV)
🏆 R² fold 3: 0.1297 | R² adj: 0.1245 (mejores hiperparámetros: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__max_iter': 400})
🔁 Fold 4 (Nested CV)
🏆 R² fold 4: 0.1586 | R² adj: 0.1536 (mejores hiperparámetros: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__max_iter': 300})
🔁 Fold 5 (Nested CV)
🏆 R² fold 5: 0.1568 | R² adj: 0.1518 (mejores hiperparámetros: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__max_iter': 400})
🔁 Fold 6 (Nested CV)
🏆 R² fold 6: 0.1613 | R² adj: 0.1563 (mejores hiperparámetros: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__max_iter': 400})
🔁 Fold 7 (Nested CV)
🏆

In [35]:
data

,Countries,Sex_1F_2M,Education,Disability,Ataxia,Bradykinesia,Barthel,Barthel_01,Diabetes_1Y_0N,Hypertension_1Y_0N,...,mci,eurocase,anydem,cogcase,prfrate,date,particid,householdid,centreid,pulse_pressure
0,Cuba,2.0,3.0,50.000000,0.0,0.0,3.75,4.3750,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,2007-12-05,2,NaN,1,61.333333
1,Cuba,1.0,4.0,16.666667,0.0,1.0,10.00,9.0625,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,2007-12-12,1,2.0,1,58.000000
2,Cuba,1.0,4.0,11.111111,0.0,1.0,10.00,8.7500,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,2008-11-03,1,6.0,1,30.000000
3,Cuba,2.0,4.0,2.777778,0.0,0.0,10.00,10.0000,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,2008-11-24,1,8.0,1,75.000000
4,Cuba,2.0,3.0,25.000000,0.0,1.0,3.75,6.5625,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,2008-11-26,2,NaN,1,49.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8386,Puerto Rico,2.0,2.0,0.000000,0.0,0.0,10.00,10.0000,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,2013-04-26,1,10262.0,20,50.000000
8387,Puerto Rico,2.0,4.0,0.000000,0.0,0.0,10.00,10.0000,1.0,1.0,...,0.0,0.0,0.0,1.0,0.0,2013-04-26,1,10263.0,20,46.666667
8388,Puerto Rico,1.0,3.0,66.666667,NaN,NaN,0.00,1.8750,1.0,1.0,...,0.0,1.0,0.0,1.0,0.0,2012-05-02,1,10265.0,20,NaN
8389,Puerto Rico,1.0,3.0,5.555556,0.0,0.0,10.00,9.3750,1.0,1.0,...,0.0,0.0,1.0,2.0,0.0,2013-05-08,1,10266.0,20,40.000000


In [36]:
def make_generic_id(row):
  h = row['householdid']
  p = row['particid']
  def to_str(val):
    if pd.isna(val): return ""
    if isinstance(val, float) and val.is_integer(): return str(int(val))
    return str(val)
  if pd.isna(h) or pd.isna(p): return np.nan
  uid_str = to_str(h) + "_" + to_str(p)
  return 'SUBJ_' + hashlib.md5(uid_str.encode()).hexdigest()[:8]

if 'particid' in data.columns and 'householdid' in data.columns:
  data['generic_id'] = data.apply(make_generic_id, axis=1)

results_df['y_pred_corrected'] = results_df['y_labels'] + results_df['GAP_corrected']
results_df = results_df.set_index('ID')

results_df = results_df.sort_index()

df_final = pd.concat([data[['generic_id', 'date', 'Age', 'anydem', 'dsmcase', 'cogcase', 'mci', 'Countries'] + best_features], results_df], axis = 1)

In [37]:
df_final.to_parquet('Results/wo-used-diag-vars-and-funct-vars_all-subjects-Incidence.parquet')

perm_imp.to_frame(name="perm_importance").to_parquet("Results/wo-used-diag-vars-and-funct-vars_all-subjects-perm_imp-Incidence.parquet")


import pickle

out = {
  "r2_scores": r2_scores,
  "r2_adj_scores": r2_adj_scores,
  "r2_global": r2_global,
  "r2_adj_global": r2_adj_global
}

with open("Results/r2_wo-used-diag-vars-and-funct-vars_all-subjects-Incidence.pkl", "wb") as f:
  pickle.dump(out, f)


### Filtering bad subjects

In [ ]:
data_filtered, non_cn_bad_subjects, r2_scores, results_df = nested_cv_flag_bad_subjects(
  data_=data,
  best_features=best_features,
  y_col="Age",
  diag_col="anydem"
)

🔁 Fold 1 (Nested CV)
🏆 R² fold 1: 0.1753
🔁 Fold 2 (Nested CV)
🏆 R² fold 2: 0.1859
🔁 Fold 3 (Nested CV)
🏆 R² fold 3: 0.1293
🔁 Fold 4 (Nested CV)
🏆 R² fold 4: 0.1626
🔁 Fold 5 (Nested CV)
🏆 R² fold 5: 0.1561
🔁 Fold 6 (Nested CV)
🏆 R² fold 6: 0.1625
🔁 Fold 7 (Nested CV)
🏆 R² fold 7: 0.1547
🔁 Fold 8 (Nested CV)
🏆 R² fold 8: 0.1813
🔁 Fold 9 (Nested CV)
🏆 R² fold 9: 0.1618
🔁 Fold 10 (Nested CV)
🏆 R² fold 10: 0.1362
🚩 Sujetos identificados (NO CN) con impacto negativo: 135
✅ Nuevo dataframe filtrado: (8256, 75)

📊 R² promedio: 0.16056484042020108 0.017099021544375585
📊 R² global: 0.16133024178901634


### BBAGs model-filtered subjects (same results as all-subjects since no subject had a negative impact)

In [ ]:
r2_scores, r2_adj_scores, results_df, shap_imp, perm_imp, r2_global, r2_adj_global = run_nested_cv_hgbr(
  data_=data_filtered,
  best_features=best_features,
  y_col="Age",
  diag_col="anydem"
)

🔁 Fold 1 (Nested CV)
🏆 R² fold 1: 0.1500 | R² adj: 0.1448 (mejores hiperparámetros: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__max_iter': 400})
🔁 Fold 2 (Nested CV)
🏆 R² fold 2: 0.1719 | R² adj: 0.1668 (mejores hiperparámetros: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__max_iter': 400})
🔁 Fold 3 (Nested CV)
🏆 R² fold 3: 0.1403 | R² adj: 0.1350 (mejores hiperparámetros: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__max_iter': 400})
🔁 Fold 4 (Nested CV)
🏆 R² fold 4: 0.1943 | R² adj: 0.1894 (mejores hiperparámetros: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__max_iter': 400})
🔁 Fold 5 (Nested CV)
🏆 R² fold 5: 0.2035 | R² adj: 0.1986 (mejores hiperparámetros: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__max_iter': 400})
🔁 Fold 6 (Nested CV)
🏆 R² fold 6: 0.1586 | R² adj: 0.1535 (mejores hiperparámetros: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__max_iter': 400})
🔁 Fold 7 (Nested CV)
🏆

In [40]:
if 'particid' in data_filtered.columns and 'householdid' in data_filtered.columns:
  data_filtered['generic_id'] = data_filtered.apply(make_generic_id, axis=1)

results_df['y_pred_corrected'] = results_df['y_labels'] + results_df['GAP_corrected']
results_df = results_df.set_index('ID')

results_df = results_df.sort_index()

df_final = pd.concat([data_filtered[['generic_id', 'date', 'Age', 'anydem', 'dsmcase', 'cogcase', 'mci', 'Countries'] + best_features], results_df], axis = 1)

In [ ]:
df_final.to_parquet('Results/wo-used-diag-vars-and-funct-vars_filtered-subjects-Incidence.parquet')
perm_imp.to_frame(name="perm_importance").to_parquet("Results/wo-used-diag-vars-and-funct-vars_filtered-subjects-perm_imp-Incidence.parquet")


out = {
  "r2_scores": r2_scores,
  "r2_adj_scores": r2_adj_scores,
  "r2_global": r2_global,
  "r2_adj_global": r2_adj_global
}

with open("Results/r2_wo-used-diag-vars-and-funct-vars_filtered-subjects-Incidence.pkl", "wb") as f:
  pickle.dump(out, f)